In [1]:
import pandas as pd
final_df=pd.read_csv(r"C:\Users\Shubham Verma\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\Ecosoul\Processed Files\Executive\final_all_geography_pnl.csv")

In [2]:
final_df = final_df.drop(columns=['platform', 'Channel','client_id'])

In [3]:
import pandas as pd

final_df['year_month'] = pd.to_datetime(
    final_df['year_month'],
    errors='coerce'
)

final_df['year'] = final_df['year_month'].dt.year
final_df['month'] = final_df['year_month'].dt.strftime('%B')
final_df['year'] = final_df['year'].astype(str)
final_df['month'] = final_df['month'].astype(str).str.zfill(2)
final_df['year_month'] = (
    final_df['month'] + '_' + final_df['year']
)

In [4]:
final_df[final_df.select_dtypes(include='float').columns] = \
    final_df.select_dtypes(include='float').round(2)

In [5]:
import pandas as pd
import urllib.parse
import re
from sqlalchemy import create_engine, text, inspect

# ======================================================
# MYSQL CONFIG
# ======================================================
MYSQL_CONFIG = {
    "host": "192.168.50.148",
    "port": 3306,
    "user": "datahive_esh_test",
    "password": "Datahive@321!",
    "database": "datahive_esh_test",
}

DEPARTMENT = "ecom"
BASE_TABLE_NAME = "pnl"
TABLE_NAME = f"{DEPARTMENT}_{BASE_TABLE_NAME}"
 
# ======================================================
# CLEAN MYSQL COLUMN NAMES
# ======================================================
def clean_mysql_column(col):
    col = col.strip()
    col = col.replace(" ", "_")
    col = col.replace("%", "pct")
    col = col.replace("-", "_")
    col = col.replace("/", "_")
    col = col.replace("\\", "_")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = col.replace("[", "")
    col = col.replace("]", "")
    col = re.sub(r"__+", "_", col)
    col = re.sub(r"[^A-Za-z0-9_]", "", col)
    return col.lower()

# ======================================================
# CREATE MYSQL ENGINE
# ======================================================
password_encoded = urllib.parse.quote_plus(MYSQL_CONFIG["password"])

engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_CONFIG['user']}:{password_encoded}"
    f"@{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}",
    pool_pre_ping=True
)

# ======================================================
# TEST MYSQL CONNECTION (OPTIONAL BUT RECOMMENDED)
# ======================================================
def test_mysql_connection():
    print("\n🔍 Testing MySQL connection...")
    try:
        with engine.connect() as conn:
            db = conn.execute(text("SELECT DATABASE();")).fetchone()
            print("✅ Connected to database:", db[0])
    except Exception as e:
        print("❌ Connection failed:", e)
        return False
    return True

# ======================================================
# SAVE final_df TO MYSQL
# ======================================================
def save_final_df_to_mysql(final_df: pd.DataFrame):
    print("\n🟦 Uploading final_df to MySQL...")

    df = final_df.copy()
    print("📊 Rows to upload:", len(df))

    # Clean column names
    df.columns = [clean_mysql_column(c) for c in df.columns]

    inspector = inspect(engine)
    table_exists = inspector.has_table(TABLE_NAME)

    try:
        with engine.begin() as conn:

            if table_exists:
                print("🗑 Truncating existing table...")
                conn.execute(text(f"TRUNCATE TABLE `{TABLE_NAME}`"))
                mode = "append"
            else:
                print("📌 Creating new table...")
                mode = "fail"

            print("⬆ Inserting rows...")
            df.to_sql(
                TABLE_NAME,
                con=conn,
                if_exists=mode,
                index=False,
                chunksize=1000,
                method="multi"
            )

        print("✅ Upload completed successfully!")

    except Exception as e:
        print("❌ Upload failed:", e)

# ======================================================
# RUN
# ======================================================
# final_df MUST already exist in memory

if test_mysql_connection():
    save_final_df_to_mysql(final_df)



🔍 Testing MySQL connection...


✅ Connected to database: datahive_esh_test

🟦 Uploading final_df to MySQL...
📊 Rows to upload: 11064


🗑 Truncating existing table...
⬆ Inserting rows...


✅ Upload completed successfully!


In [6]:
import pandas as pd
final_df=pd.read_csv(r"C:\Users\Shubham Verma\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\Ecosoul\Processed Files\finance\Retail_AR_Wroking.csv")

In [7]:
final_df.columns

Index(['Invoice Date', 'Month', 'Invoice No.', 'PO Number', 'Party Name',
       'Factored', 'Gross Amount', 'Allowances', 'Payment Discount', 'Rate',
       'Net Amount', 'Retailer Charges', 'Retailer paid', 'AR for invoice',
       'AR total', 'Payment from Retailer', 'Payment Date', 'Date o/s',
       'Payment Days', 'Cheque Date',
       'Payment Discount if payment received within days', 'Due after days',
       'Payment Terms', 'payment_expected_date', 'status', 'days_gap', 'year'],
      dtype='object')

In [8]:
import pandas as pd
import urllib.parse
import re
from sqlalchemy import create_engine, text, inspect

# ======================================================
# MYSQL CONFIG
# ======================================================
MYSQL_CONFIG = {
    "host": "192.168.50.148",
    "port": 3306,
    "user": "datahive_esh_test",
    "password": "Datahive@321!",
    "database": "datahive_esh_test",
}

DEPARTMENT = "finance"
BASE_TABLE_NAME = "ar_working"
TABLE_NAME = f"{DEPARTMENT}_{BASE_TABLE_NAME}"
 
# ======================================================
# CLEAN MYSQL COLUMN NAMES
# ======================================================
def clean_mysql_column(col):
    col = col.strip()
    col = col.replace(" ", "_")
    col = col.replace("%", "pct")
    col = col.replace("-", "_")
    col = col.replace("/", "_")
    col = col.replace("\\", "_")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = col.replace("[", "")
    col = col.replace("]", "")
    col = re.sub(r"__+", "_", col)
    col = re.sub(r"[^A-Za-z0-9_]", "", col)
    return col.lower()

# ======================================================
# CREATE MYSQL ENGINE
# ======================================================
password_encoded = urllib.parse.quote_plus(MYSQL_CONFIG["password"])

engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_CONFIG['user']}:{password_encoded}"
    f"@{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}",
    pool_pre_ping=True
)

# ======================================================
# TEST MYSQL CONNECTION (OPTIONAL BUT RECOMMENDED)
# ======================================================
def test_mysql_connection():
    print("\n🔍 Testing MySQL connection...")
    try:
        with engine.connect() as conn:
            db = conn.execute(text("SELECT DATABASE();")).fetchone()
            print("✅ Connected to database:", db[0])
    except Exception as e:
        print("❌ Connection failed:", e)
        return False
    return True

# ======================================================
# SAVE final_df TO MYSQL
# ======================================================
def save_final_df_to_mysql(final_df: pd.DataFrame):
    print("\n🟦 Uploading final_df to MySQL...")

    df = final_df.copy()
    print("📊 Rows to upload:", len(df))

    # Clean column names
    df.columns = [clean_mysql_column(c) for c in df.columns]

    inspector = inspect(engine)
    table_exists = inspector.has_table(TABLE_NAME)

    try:
        with engine.begin() as conn:

            if table_exists:
                print("🗑 Truncating existing table...")
                conn.execute(text(f"TRUNCATE TABLE `{TABLE_NAME}`"))
                mode = "append"
            else:
                print("📌 Creating new table...")
                mode = "fail"

            print("⬆ Inserting rows...")
            df.to_sql(
                TABLE_NAME,
                con=conn,
                if_exists=mode,
                index=False,
                chunksize=1000,
                method="multi"
            )

        print("✅ Upload completed successfully!")

    except Exception as e:
        print("❌ Upload failed:", e)

# ======================================================
# RUN
# ======================================================
# final_df MUST already exist in memory

if test_mysql_connection():
    save_final_df_to_mysql(final_df)



🔍 Testing MySQL connection...


✅ Connected to database: datahive_esh_test

🟦 Uploading final_df to MySQL...
📊 Rows to upload: 4582


🗑 Truncating existing table...
⬆ Inserting rows...


❌ Upload failed: (mysql.connector.errors.DataError) 1366 (22007): Incorrect double value: '$0.00' for column `datahive_esh_test`.`finance_ar_working`.`gross_amount` at row 540
[SQL: INSERT INTO finance_ar_working (invoice_date, month, invoice_no, po_number, party_name, factored, gross_amount, allowances, payment_discount, rate, net_amount, retailer_charges, retailer_paid, ar_for_invoice, ar_total, payment_from_retailer, payment_date, date_o_s, payment_days, cheque_date, payment_discount_if_payment_received_within_days, due_after_days, payment_terms, payment_expected_date, status, days_gap, year) VALUES (%(invoice_date_m0)s, %(month_m0)s, %(invoice_no_m0)s, %(po_number_m0)s, %(party_name_m0)s, %(factored_m0)s, %(gross_amount_m0)s, %(allowances_m0)s, %(payment_discount_m0)s, %(rate_m0)s, %(net_amount_m0)s, %(retailer_charges_m0)s, %(retailer_paid_m0)s, %(ar_for_invoice_m0)s, %(ar_total_m0)s, %(payment_from_retailer_m0)s, %(payment_date_m0)s, %(date_o_s_m0)s, %(payment_days_m0)s, %(cheque

In [9]:
# final_df = pd.read_csv(r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\template_data_team\usa\output\competitor_df.csv")

In [10]:
# import pandas as pd
# import urllib.parse
# import re
# from sqlalchemy import create_engine, text, inspect

# # ======================================================
# # MYSQL CONFIG
# # ======================================================
# MYSQL_CONFIG = {
#     "host": "192.168.50.148",
#     "port": 3306,
#     "user": "datahive_esh_test",
#     "password": "Datahive@321!",
#     "database": "datahive_esh_test",
# }

# DEPARTMENT = "template"
# BASE_TABLE_NAME = "competitor"
# TABLE_NAME = f"{DEPARTMENT}_{BASE_TABLE_NAME}"
 
# # ======================================================
# # CLEAN MYSQL COLUMN NAMES
# # ======================================================
# def clean_mysql_column(col):
#     col = col.strip()
#     col = col.replace(" ", "_")
#     col = col.replace("%", "pct")
#     col = col.replace("-", "_")
#     col = col.replace("/", "_")
#     col = col.replace("\\", "_")
#     col = col.replace("(", "")
#     col = col.replace(")", "")
#     col = col.replace("[", "")
#     col = col.replace("]", "")
#     col = re.sub(r"__+", "_", col)
#     col = re.sub(r"[^A-Za-z0-9_]", "", col)
#     return col.lower()

# # ======================================================
# # CREATE MYSQL ENGINE
# # ======================================================
# password_encoded = urllib.parse.quote_plus(MYSQL_CONFIG["password"])

# engine = create_engine(
#     f"mysql+mysqlconnector://{MYSQL_CONFIG['user']}:{password_encoded}"
#     f"@{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}",
#     pool_pre_ping=True
# )

# # ======================================================
# # TEST MYSQL CONNECTION (OPTIONAL BUT RECOMMENDED)
# # ======================================================
# def test_mysql_connection():
#     print("\n🔍 Testing MySQL connection...")
#     try:
#         with engine.connect() as conn:
#             db = conn.execute(text("SELECT DATABASE();")).fetchone()
#             print("✅ Connected to database:", db[0])
#     except Exception as e:
#         print("❌ Connection failed:", e)
#         return False
#     return True

# # ======================================================
# # SAVE final_df TO MYSQL
# # ======================================================
# def save_final_df_to_mysql(final_df: pd.DataFrame):
#     print("\n🟦 Uploading final_df to MySQL...")

#     df = final_df.copy()
#     print("📊 Rows to upload:", len(df))

#     # Clean column names
#     df.columns = [clean_mysql_column(c) for c in df.columns]

#     inspector = inspect(engine)
#     table_exists = inspector.has_table(TABLE_NAME)

#     try:
#         with engine.begin() as conn:

#             if table_exists:
#                 print("🗑 Truncating existing table...")
#                 conn.execute(text(f"TRUNCATE TABLE `{TABLE_NAME}`"))
#                 mode = "append"
#             else:
#                 print("📌 Creating new table...")
#                 mode = "fail"

#             print("⬆ Inserting rows...")
#             df.to_sql(
#                 TABLE_NAME,
#                 con=conn,
#                 if_exists=mode,
#                 index=False,
#                 chunksize=1000,
#                 method="multi"
#             )

#         print("✅ Upload completed successfully!")

#     except Exception as e:
#         print("❌ Upload failed:", e)

# # ======================================================
# # RUN
# # ======================================================
# # final_df MUST already exist in memory

# if test_mysql_connection():
#     save_final_df_to_mysql(final_df)
